# 08 · Aprendizaje supervisado en 8 pasos

**Bloque de IA · Sesión 2 · Parte 2.** Recorremos los ocho pasos de un proyecto supervisado con datos reales: el valor de la vivienda en 20.640 zonas de California (censo de EE. UU. de 1990), el conjunto *California Housing* que viene con scikit-learn.

Funciona en Google Colab sin subir nada: **Entorno de ejecución → Ejecutar todas**. En local: `pip install scikit-learn pandas matplotlib joblib`.

## Paso 1 · Elegir algoritmo

Queremos predecir un **número** (el valor de la vivienda): es un problema de **regresión**. Probaremos dos candidatos: regresión lineal (sencilla) y árbol de decisión (más flexible), y al final un random forest.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

## Paso 2 · Seleccionar datos

Cada fila es una zona del censo. `MedInc` es la renta mediana (en decenas de miles de dólares) y la etiqueta `MedHouseVal` es el valor mediano de la vivienda (en cientos de miles de dólares; está limitado a 5, es decir, 500.000 $).

In [ ]:
datos = fetch_california_housing(as_frame=True).frame
print(datos.shape)
datos.iloc[[0, 1000, 15000, 20000]]

In [ ]:
# Calidad del dato: nulos y valores extremos
print("Nulos por columna:", datos.isna().sum().sum())
print("Zonas en el tope de 500.000 $:", round((datos["MedHouseVal"] >= 5).mean() * 100, 1), "%")
datos.describe().round(2)

In [ ]:
X = datos.drop(columns="MedHouseVal")   # variables de entrada
y = datos["MedHouseVal"]                # etiqueta

## Paso 3 · Construir el modelo

`fit()` busca los pesos que hacen mínimo el error al cuadrado con los datos de **entrenamiento**. Antes repartimos los datos (lo explica el paso 4).

In [ ]:
# Reparto 70 / 15 / 15: entrenamiento, validación y test
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_ent, X_val, y_ent, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15 / 0.85, random_state=42)
print(len(X_ent), len(X_val), len(X_test))

In [ ]:
lineal = LinearRegression().fit(X_ent, y_ent)
pd.Series(lineal.coef_, index=X.columns).round(3)

Lectura: un peso de 0,44 en `MedInc` significa que, con el resto igual, cada 10.000 $ más de renta en la zona suman unos 44.000 $ al valor.

El árbol se construye de otra forma: busca la pregunta que más reduce el error y repite en cada rama.

In [ ]:
arbol2 = DecisionTreeRegressor(max_depth=2, random_state=42).fit(X_ent, y_ent)
print(export_text(arbol2, feature_names=list(X.columns)))

## Paso 4 · Validar

Medimos el **error absoluto medio** (MAE) con datos de validación, que el modelo no ha visto. Lo comparamos con una **línea base**: predecir siempre la media.

In [ ]:
def mae_dolares(modelo, X_, y_):
    return round(mean_absolute_error(y_, modelo.predict(X_)) * 100_000)

base = DummyRegressor().fit(X_ent, y_ent)
arbol6 = DecisionTreeRegressor(max_depth=6, random_state=42).fit(X_ent, y_ent)
for nombre, m in [("Línea base", base), ("Regresión lineal", lineal), ("Árbol, 6 niveles", arbol6)]:
    print(f"{nombre:18s} {mae_dolares(m, X_val, y_val):>8,} $")

## Paso 5 · Ajustar parámetros

El número de niveles del árbol (`max_depth`) lo elegimos nosotros. Probamos varios y nos quedamos con el menor error de **validación**. Fíjate en el sobreajuste: el error de entrenamiento baja a cero, pero el de validación vuelve a subir.

In [ ]:
filas = []
for niveles in [2, 4, 6, 8, 10, 15, None]:
    a = DecisionTreeRegressor(max_depth=niveles, random_state=42).fit(X_ent, y_ent)
    filas.append({"niveles": niveles or "sin límite",
                  "error entrenamiento ($)": mae_dolares(a, X_ent, y_ent),
                  "error validación ($)": mae_dolares(a, X_val, y_val)})
tabla = pd.DataFrame(filas)
tabla

In [ ]:
tabla.plot(x="niveles", y=["error entrenamiento ($)", "error validación ($)"], marker="o", title="Error según los niveles del árbol")
plt.show()

In [ ]:
# Un bosque de 200 árboles suele mejorar a un solo árbol (tarda unos segundos)
bosque = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42).fit(X_ent, y_ent)
print("Random forest, validación:", f"{mae_dolares(bosque, X_val, y_val):,} $")

## Paso 6 · Probar con datos nuevos

Reentrenamos con entrenamiento + validación y medimos **una sola vez** con test. Si después retocáramos el modelo mirando este número, dejaría de ser honesto.

In [ ]:
X_final, y_final = pd.concat([X_ent, X_val]), pd.concat([y_ent, y_val])
base_f = DummyRegressor().fit(X_final, y_final)
lineal_f = LinearRegression().fit(X_final, y_final)
bosque_f = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42).fit(X_final, y_final)
for nombre, m in [("Línea base", base_f), ("Regresión lineal", lineal_f), ("Random forest", bosque_f)]:
    print(f"{nombre:18s} {mae_dolares(m, X_test, y_test):>8,} $")

In [ ]:
pd.Series(bosque_f.feature_importances_, index=X.columns).sort_values().plot.barh(title="Qué pesa más en el random forest")
plt.show()

## Paso 7 · Implementar

Guardamos el modelo para usarlo en otro proceso (por lotes cada noche o detrás de una API). En Databricks se registraría con MLflow, junto con los datos y parámetros que lo produjeron.

In [ ]:
import joblib
joblib.dump(bosque_f, "valor_vivienda_v1.joblib")

modelo = joblib.load("valor_vivienda_v1.joblib")
nuevas = X_test.head(3)
pd.DataFrame({"predicción ($)": (modelo.predict(nuevas) * 100_000).round(-3),
              "valor real ($)": (y_test.head(3) * 100_000).round(-3)})

## Paso 8 · Optimizar

En producción hay que vigilar el modelo: comparar con los precios reales cuando se conocen, detectar si los datos de entrada cambian (deriva) y reentrenar. Este modelo está entrenado con datos de **1990**: aplicado hoy daría valores absurdos, el caso extremo de deriva. Es lo que le pasó, a otra escala, a Zillow en 2021.

### Para practicar
1. Cambia `random_state` del reparto y comprueba cuánto varían los errores.
2. Quita las columnas `Latitude` y `Longitude`: ¿cuánto empeora el modelo?
3. Prueba `max_depth` de 8 a 12 en el árbol y `n_estimators` de 50 a 400 en el bosque.